#### RAG-Workflow Schritt für Schritt

Struktur: 

1. Websraping der Urteilstexte 
2. Abspeichern der Urteilstexte in Pandas Dataframe
3. Vektorisieren der Textinhalte
4. Absrufmechanismus erstellen

Import der notwendigen Libraries und Konfiguration wichtiger Variablen:

In [1]:
from __future__ import annotations

import math
from pathlib import Path
from typing import List
import pandas as pd

import ollama
import requests
from bs4 import BeautifulSoup

EMBED_MODEL = "qwen3-embedding:8b" #alternativen: 0.6b, 4b und 8b
CHAT_MODEL = "gemma3:12b"
OUTPUT_PATH = Path("bverfg_urteil_qwen.pkl")

Funktionen zum Webscraping und Parsing des Html-Codes: 

In [2]:

def fetch_html(url: str, timeout: int = 30) -> str:
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.text


def parse_bverfg_decision(url: str) -> pd.DataFrame:
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    reasons_div = soup.find("div", class_="c-decision__reasons")
    if reasons_div is None:
        raise ValueError("Kein <div class='c-decision__reasons'> gefunden.")

    rows = []
    current_randnummer = None

    for tag in reasons_div.children:
        # Nur echte HTML-Tags verarbeiten
        if not getattr(tag, "name", None):
            continue

        # Randnummern: <p class="is-anchor" id="8">8</p>
        if tag.name == "p" and "is-anchor" in (tag.get("class") or []):
            rn_text = tag.get_text(strip=True)
            if rn_text.isdigit():
                current_randnummer = int(rn_text)
            else:
                current_randnummer = None
            continue

        # Absatztext: <p class="justify">...</p>
        if (
            tag.name == "p"
            and "justify" in (tag.get("class") or [])
            and current_randnummer is not None
        ):
            text = tag.get_text(" ", strip=True)
            rows.append(
                {
                    "Text": text,
                    "Embeddings": None,
                    "Randnummer": current_randnummer,
                    "url": url,
                }
            )

    if not rows:
        raise ValueError(
            "Keine Randnummer/Text-Paare in <div class='c-decision__reasons'> gefunden."
        )

    return pd.DataFrame(rows, columns=["Text", "Embeddings", "Randnummer", "url"])


In [3]:
#debugging: Test der in der vorangegangenen Zelle definierten Funktion zum Scrapen und Parsen der BVerfG-Entscheidung
url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2025/03/rs20250326_2bvr150520.html"
df = parse_bverfg_decision(url)
df

,Text,Embeddings,Randnummer,url
0,Die Verfassungsbeschwerde der sechs Beschwerde...,None,1,https://www.bundesverfassungsgericht.de/Shared...
1,1. Der – auch heute noch erhobene – Solidaritä...,None,2,https://www.bundesverfassungsgericht.de/Shared...
2,2. Der Zuschlagsatz zur Einkommen- oder Körper...,None,3,https://www.bundesverfassungsgericht.de/Shared...
3,3. Abgabepflichtig sind nach § 2 SolZG 1995 zu...,None,4,https://www.bundesverfassungsgericht.de/Shared...
4,4. Durch das Gesetz zur Rückführung des Solida...,None,5,https://www.bundesverfassungsgericht.de/Shared...
...,...,...,...,...
172,(c) Der Gesetzgeber hat Art. 3 Abs. 1 GG auch ...,None,173,https://www.bundesverfassungsgericht.de/Shared...
173,Wie im Falle der Kapitalertragsteuer folgt die...,None,174,https://www.bundesverfassungsgericht.de/Shared...
174,Die Berücksichtigung sozialer Gesichtspunkte i...,None,175,https://www.bundesverfassungsgericht.de/Shared...
175,"Das SolZG 1995 verletzt, wie sich im Rahmen de...",None,176,https://www.bundesverfassungsgericht.de/Shared...


Funktionen zur Vektorisierung der Urteils-Absätze im Dataframe

In [ ]:

def embed_texts(texts: list[str], model: str = EMBED_MODEL) -> list[list[float]]:
    """
    Erzeugt Embeddings über die Ollama-Python-Library.
    """
    response = ollama.embed(model=model, input=texts)

    if "embeddings" not in response:
        raise ValueError(f"Unerwartete Ollama-Antwort: {response}")

    embeddings = response["embeddings"]

    if len(embeddings) != len(texts):
        raise ValueError(
            f"Anzahl Embeddings ({len(embeddings)}) passt nicht zu "
            f"Anzahl Texte ({len(texts)})."
        )

    return embeddings

def add_embeddings_to_df(
    df: pd.DataFrame,
    model: str = EMBED_MODEL,
    batch_size: int = 32,
) -> pd.DataFrame:
    """
    Fügt einem DataFrame in Batches Embeddings hinzu.
    """
    df = df.copy()
    texts = df["Text"].tolist()

    all_embeddings: list[list[float]] = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        batch_embeddings = embed_texts(batch, model=model)
        all_embeddings.extend(batch_embeddings)

    df["Embeddings"] = all_embeddings
    return df

def save_dataframe(df: pd.DataFrame, path: Path = OUTPUT_PATH) -> None:
    """
    Pickle ist hier praktisch, weil Embeddings Listen von Floats sind.
    """
    df.to_pickle(path)

def load_dataframe(path: Path = OUTPUT_PATH) -> pd.DataFrame:
    return pd.read_pickle(path)

def build_rag_dataframe_from_url(
    url: str,
    model: str = EMBED_MODEL,
    batch_size: int = 32,
) -> pd.DataFrame:
    """
    Komplettpipeline:
    1. URL laden
    2. Absätze parsen
    3. Embeddings erzeugen
    """
    df = parse_bverfg_decision(url)
    df = add_embeddings_to_df(df, model=model, batch_size=batch_size)
    save_dataframe(df)
    return df

def get_or_build_rag_dataframe(
    url: str,
    model: str = EMBED_MODEL,
    batch_size: int = 32,
    path: Path = OUTPUT_PATH,
) -> pd.DataFrame:
    if path.exists():
        try:
            df = load_dataframe(path)

            if "url" in df.columns and not df.empty:
                gespeicherte_url = df["url"].iloc[0]

                if gespeicherte_url == url:
                    print("Urteil bereits vorhanden – lade DataFrame aus Pickle.")
                    return df

            print("Andere URL im Pickle gefunden – scrape und embedde neu.")
        except Exception as e:
            print(f"Pickle konnte nicht geladen werden ({e}) – scrape und embedde neu.")

    else:
        print("Noch kein Pickle vorhanden – scrape und embedde neu.")

    return build_rag_dataframe_from_url(url, model=model, batch_size=batch_size)


In [ ]:
#debugging: Test der Komplettpipeline zum Laden, Parsen, Vektorisieruen und Speichern der BVerfG-Entscheidung
url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2025/03/rs20250326_2bvr150520.html"
df = get_or_build_rag_dataframe(url, model=EMBED_MODEL, batch_size=16)
df

Funktionen zum semantischen Ähnlichkeitsvergleich: 

In [ ]:

def load_dataframe(path: Path = OUTPUT_PATH) -> pd.DataFrame:
    return pd.read_pickle(path)

def cosine_similarity(a: List[float], b: List[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def search_similar_passages(
    query: str,
    df: pd.DataFrame,
    model: str = EMBED_MODEL,
    top_k: int = 2,
) -> pd.DataFrame:
    if "Embeddings" not in df.columns:
        raise ValueError("Die Spalte 'Embeddings' fehlt.")

    if df["Embeddings"].isna().any():
        raise ValueError("Es fehlen Embeddings im DataFrame.")

    query_embedding = embed_texts([query], model=model)[0]

    result_df = df.copy()
    result_df["Score"] = result_df["Embeddings"].apply(
        lambda emb: cosine_similarity(query_embedding, emb)
    )

    result_df = (
        result_df.sort_values("Score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    return result_df

In [ ]:
# debugging: Test der Ähnlichkeitssuche
query = "verstößt das SolZG gegen das Grundgesetz?"
url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2025/03/rs20250326_2bvr150520.html"
df = get_or_build_rag_dataframe(url, model=EMBED_MODEL, batch_size=16)

hits = search_similar_passages(query, df, model=EMBED_MODEL, top_k=10)
print("\nTop-Treffer:")
hits # hits ist ein DataFrame mit den ähnlichsten Absätzen, sortiert nach Score (siehe return-Wert der Funktion search_similar_passages)

Zusammenbau des Prompt-Strings aus: 
1. Template 
2. Suchergebnissen

In [ ]:
def build_context_from_hits(hits: pd.DataFrame) -> str:
    """
    Baut aus den Top-Hits einen sauberen Kontextblock für das Sprachmodell.
    """
    context_parts = []

    for i, row in hits.iterrows():
        randnummer = row["Randnummer"]
        text = row["Text"]
        score = row.get("Score", None)

        part = (
            f"[Quelle {i + 1} | Randnummer {randnummer}"
            + (f" | Score {score:.4f}" if score is not None else "")
            + "]\n"
            f"{text}"
        )
        context_parts.append(part)

    return "\n\n".join(context_parts)


def build_rag_prompt(query: str, hits: pd.DataFrame) -> str:
    """
    Erstellt den Prompt für das Sprachmodell.
    """
    context = build_context_from_hits(hits)

    prompt = f"""
Du bist ein ein Experte für das deutsche Verfassungsrecht und spezialisiert auf die Analyse von Entscheidungen des Bundesverfassungsgerichts.

Du erhältst nachfolgend eine Frage sowie relevante Fundstellen aus einer Entscheidung des Bundesverfassungsgerichts (BVerfG), die du für die Beantwortung der Frage verwenden musst.
Beantworte die nachfolgend gestellte Frage ausschließlich auf Grundlage der unten angegebenen Fundstellen.
Wenn die Fundstellen für eine sichere Antwort nicht ausreichen, sage das ausdrücklich.
Zitiere in deiner Antwort möglichst die Randnummern in Klammern, z. B. (Rn. 12).
Beachte, dass Fundstellen in denen rechtsauffassungen im Konjunktiv dargestellt werden, nicht zwingend die Auffassung des Gerichts widerspiegeln, sondern auch Auffassung einer der Parteien sein können.

Frage:
{query}

Relevante Fundstellen:
{context}

Antworte präzise, juristisch sauber und gut strukturiert.
"""
    return prompt.strip()



In [ ]:
# debugging: Test der Komplettpipeline zum Laden, Parsen, Vektorisieren, Ähnlichketsabgleich und schließlich der Prompt-Erstellung
query = "verstößt das SolZG gegen das Grundgesetz?"
hits = search_similar_passages(query, df, model=EMBED_MODEL, top_k=10)
prompt = build_rag_prompt(query, hits)
print("\n=== PROMPT ===")
print(prompt)

Zusammenbau der Gesamt-RAG-Pipeline in einer Funktion ``` ask_rag() ```

In [ ]:
def ask_rag(
    query: str,
    url: str,
    retrieval_model: str = EMBED_MODEL,
    chat_model: str = CHAT_MODEL,
    top_k: int = 5,
) -> dict:
    """
    Führt Retrieval + Promptaufbau + Antwortgenerierung aus.
    Streamt die Antwort live und gibt am Ende weiterhin alles gesammelt zurück.
    """
    hits = search_similar_passages(
        query=query,
        df=get_or_build_rag_dataframe(url=url, model=retrieval_model),
        model=retrieval_model,
        top_k=top_k,
    )

    prompt = build_rag_prompt(query, hits)

    stream = ollama.chat(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Du beantwortest Fragen zu BVerfG-Entscheidungen "
                    "ausschließlich auf Basis des bereitgestellten Kontexts."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        stream=True,
    )

    answer_parts = []

    for chunk in stream:
        content = chunk["message"]["content"]
        print(content, end="", flush=True)
        answer_parts.append(content)

    answer = "".join(answer_parts)

    return {
        "query": query,
        "answer": answer,
        "hits": hits,
        "prompt": prompt,
    }

In [ ]:
query = "verstößt das SolZG gegen Art. 14 GG?"
#query = "Unter welchen Voraussetzungen darf eine Ergänzungsabgabe nach Art. 106 Abs. 1 Nr. 6 GG erhoben werden?"
url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2025/03/rs20250326_2bvr150520.html"

result = ask_rag(query, url, top_k=5)

print("=== ANTWORT ===")
print(result["answer"])

print("\n=== TOP-5-HITS ===")
print(result["hits"][["Randnummer", "Score", "Text"]].to_string(index=False))